In [2]:
import os

In [3]:
pwd = os.getcwd()

In [4]:
os.chdir("../")
print(os.getcwd())

/Users/xenan.bilgin/Projects/MLops/mlops-project1-wine-quality


In [5]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/xenablgn/mlops-project1-wine-quality.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="xenablgn"
os.environ["MLFLOW_TRACKING_PASSWORD"]="604e97600e56ba5171d25414da5df88c91584267"


In [ ]:

from attr import dataclass


@dataclass 
class ModelEvaluationConfig:
    root_dir: str
    test_data_path: str
    model_path: str
    all_params: dict
    metric_file_name : str
    target_column: str
    mlflow_uri: str

    
    

In [ ]:
from src.mlops1_data_science_project import logger
from src.mlops1_data_science_project.constants import (
    CONFIG_FILE_PATH,
    PARAMS_FILE_PATH,
    SCHEMA_FILE_PATH,
)
from src.mlops1_data_science_project.utils.common import create_directories, read_yaml

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_eval_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN.name
        create_directories([config.root_dir])

        model_eval_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema,
            mlflow_uri= os.environ["MLFLOW_TRACKING_URI"]
        )

        return model_eval_config

In [ ]:
from pathlib import Path
from urllib.parse import urlparse

import joblib
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import mlflow

from mlops1_data_science_project.utils.common import save_json


class ModelEvaluator:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, y_true, y_pred):
        rmse = mean_squared_error(y_true, y_pred, squared=False)
        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)

        return rmse, r2, mae

    def log_mlflow(self):
        model = joblib.load(self.config.model_path)
        test_df = pd.read_csv(self.config.test_data_path)

        text_X= test_df.drop(self.config.target_column, axis=1)
        text_y = test_df[self.config.target_column]

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse.urlparse(mlflow.get_tracking_uri()).scheme

        mlflow.set_experiment("Wine Quality Prediction")
    

        with mlflow.start_run(run_name="Model Evaluation"):
            
            y_pred = model.predict(text_X)

            rmse, r2, mae = self.eval_metrics(text_y, y_pred)
            scores = {
                "rmse": rmse,
                "r2": r2,
                "mae": mae
            }
            save_json(path=Path(self.config.metric_file_name), data=scores)


            mlflow.log_param("model_path", self.config.model_path)
            mlflow.log_param("test_data_path", self.config.test_data_path)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", register_model_name="WineQualityModel")
            else:
                mlflow.sklearn.log_model(model, "model")
                

        


In [ ]:
try:
    config = ConfigurationManager().get_model_eval_config()
    model_evaluator = ModelEvaluator(config)
    model_evaluator.log_mlflow()
except Exception as e:
    logger.exception(e)